In [5]:
import xgboost as xgb

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from xgboost import XGBClassifier

In [7]:
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / "model_training_table.csv")

print(df.shape)
df.head()

(3000, 67)


,vehicle_id,year,mileage,vehicle_age,is_electric,is_hybrid,is_diesel,avg_engine_temp,max_engine_temp,std_engine_temp,...,model_Maverick,model_Super Duty,model_Transit,fuel_type_Electric,fuel_type_Gasoline,fuel_type_Hybrid,industry_Delivery,industry_Government,industry_Logistics,industry_Utilities
0,V00001,2018,125198,8,0,1,0,94.481981,106.062108,5.035404,...,False,False,True,False,False,True,False,False,False,False
1,V00002,2020,79155,6,0,0,0,91.018243,107.257515,5.280720,...,False,False,False,False,True,False,False,True,False,False
2,V00003,2020,126449,6,1,0,0,95.316620,109.442020,5.044226,...,True,False,False,True,False,False,False,False,False,True
3,V00004,2021,81700,5,0,0,1,91.501110,103.218983,4.403340,...,False,False,False,False,False,False,True,False,False,False
4,V00005,2022,11955,4,0,1,0,85.104323,100.018771,5.318115,...,False,True,False,False,False,True,False,False,False,True


In [8]:
target_col = "failure_next_30_days"

drop_cols = [
    "vehicle_id",
    target_col
]

X = df.drop(columns=drop_cols)
y = df[target_col]

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts(normalize=True))

X shape: (3000, 65)
y distribution:
failure_next_30_days
0    0.75
1    0.25
Name: proportion, dtype: float64


In [9]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full
)

print("Train:", X_train.shape, y_train.mean())
print("Validation:", X_val.shape, y_val.mean())
print("Test:", X_test.shape, y_test.mean())

Train: (1800, 65) 0.25
Validation: (600, 65) 0.25
Test: (600, 65) 0.25


In [10]:
def recall_at_k(y_true, y_scores, k=0.10):
    """
    Measures how many true failures are captured in the top k% riskiest vehicles.
    """
    data = pd.DataFrame({
        "y_true": y_true,
        "score": y_scores
    })

    data = data.sort_values("score", ascending=False)

    top_k_n = int(len(data) * k)
    top_k = data.head(top_k_n)

    total_failures = data["y_true"].sum()
    captured_failures = top_k["y_true"].sum()

    return captured_failures / total_failures

In [11]:
def evaluate_model(name, model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_eval, y_pred),
        "precision": precision_score(y_eval, y_pred),
        "recall": recall_score(y_eval, y_pred),
        "f1": f1_score(y_eval, y_pred),
        "roc_auc": roc_auc_score(y_eval, y_proba),
        "pr_auc": average_precision_score(y_eval, y_proba),
        "recall_at_10pct": recall_at_k(y_eval, y_proba, k=0.10),
        "recall_at_20pct": recall_at_k(y_eval, y_proba, k=0.20)
    }

    return metrics

In [ ]:
log_reg = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

log_reg.fit(X_train, y_train)

log_reg_metrics = evaluate_model(
    "Logistic Regression",
    log_reg,
    X_val,
    y_val
)

log_reg_metrics

{'model': 'Logistic Regression',
 'accuracy': 0.8266666666666667,
 'precision': 0.6116504854368932,
 'recall': 0.84,
 'f1': 0.7078651685393258,
 'roc_auc': 0.914888888888889,
 'pr_auc': 0.82320227893615,
 'recall_at_10pct': np.float64(0.38),
 'recall_at_20pct': np.float64(0.64)}

In [23]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=8,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_metrics = evaluate_model(
    "Random Forest",
    rf,
    X_val,
    y_val
)

rf_metrics

{'model': 'Random Forest',
 'accuracy': 0.8033333333333333,
 'precision': 0.5740740740740741,
 'recall': 0.8266666666666667,
 'f1': 0.6775956284153005,
 'roc_auc': 0.8847703703703703,
 'pr_auc': 0.7612306245019287,
 'recall_at_10pct': np.float64(0.34),
 'recall_at_20pct': np.float64(0.58)}

In [15]:
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb.fit(X_train, y_train)

gb_metrics = evaluate_model(
    "Gradient Boosting",
    gb,
    X_val,
    y_val
)

gb_metrics

{'model': 'Gradient Boosting',
 'accuracy': 0.855,
 'precision': 0.7404580152671756,
 'recall': 0.6466666666666666,
 'f1': 0.6903914590747331,
 'roc_auc': 0.9046222222222223,
 'pr_auc': 0.7891085953643,
 'recall_at_10pct': np.float64(0.36666666666666664),
 'recall_at_20pct': np.float64(0.6066666666666667)}

In [19]:
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

xgb_metrics = evaluate_model(
    "XGBoost",
    xgb,
    X_val,
    y_val
)

xgb_metrics

{'model': 'XGBoost',
 'accuracy': 0.855,
 'precision': 0.7266187050359713,
 'recall': 0.6733333333333333,
 'f1': 0.698961937716263,
 'roc_auc': 0.9027851851851852,
 'pr_auc': 0.7930033446579676,
 'recall_at_10pct': np.float64(0.37333333333333335),
 'recall_at_20pct': np.float64(0.6)}

In [20]:
results = pd.DataFrame([
    log_reg_metrics,
    rf_metrics,
    gb_metrics,
    xgb_metrics
]).sort_values("recall_at_10pct", ascending=False)

results

,model,accuracy,precision,recall,f1,roc_auc,pr_auc,recall_at_10pct,recall_at_20pct
0,Logistic Regression,0.826667,0.611650,0.840000,0.707865,0.914889,0.823202,0.380000,0.640000
3,XGBoost,0.855000,0.726619,0.673333,0.698962,0.902785,0.793003,0.373333,0.600000
2,Gradient Boosting,0.855000,0.740458,0.646667,0.690391,0.904622,0.789109,0.366667,0.606667
1,Random Forest,0.795000,0.561086,0.826667,0.668464,0.882978,0.760864,0.353333,0.580000


In [24]:
best_model_name = results.iloc[0]["model"]

model_map = {
    "Logistic Regression": log_reg,
    "Random Forest": rf,
    "Gradient Boosting": gb,
    "XGBoost": xgb
}

best_model = model_map[best_model_name]

print("Best model:", best_model_name)

Best model: Logistic Regression


In [25]:
y_test_pred = best_model.predict(X_test)
y_test_proba = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_test_pred)

cm

array([[366,  84],
       [ 25, 125]])

In [26]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.94      0.81      0.87       450
           1       0.60      0.83      0.70       150

    accuracy                           0.82       600
   macro avg       0.77      0.82      0.78       600
weighted avg       0.85      0.82      0.83       600



In [27]:
test_predictions = df.loc[X_test.index, ["vehicle_id"]].copy()

test_predictions["failure_probability"] = y_test_proba
test_predictions["actual_failure"] = y_test.values

test_predictions["risk_level"] = pd.cut(
    test_predictions["failure_probability"],
    bins=[0, 0.33, 0.66, 1.0],
    labels=["Low", "Medium", "High"]
)

test_predictions = test_predictions.sort_values(
    "failure_probability",
    ascending=False
)

test_predictions.head(20)

,vehicle_id,failure_probability,actual_failure,risk_level
2414,V02415,0.999718,1,High
2984,V02985,0.999265,1,High
2170,V02171,0.997787,1,High
531,V00532,0.997225,1,High
2620,V02621,0.997207,1,High
2212,V02213,0.996770,1,High
1053,V01054,0.996310,1,High
2208,V02209,0.996178,1,High
1759,V01760,0.995870,1,High
100,V00101,0.995537,1,High
